# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Explore basic metadata
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id` fields.

In [ ]:
# List all record set @id values and their fields
recordsets = list(dataset.record_sets)
print(f"There are {len(recordsets)} record sets.\n")
for rs in recordsets:
    print(f"Record set: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Fields and corresponding @id:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) type={field.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Load all record sets into DataFrames and print their columns
dataframes = {}
for rs in recordsets:
    # Use the record set ID, per Croissant
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded record set @id: {rs.id} with {df.shape[0]} records and columns: {df.columns.tolist()}")
    print("")

# For demonstration, pick the first record set for further analysis
chosen_rs_id = recordsets[0].id
print(f"Using record set '@id': {chosen_rs_id}\nColumns: {dataframes[chosen_rs_id].columns.tolist()}")
dataframes[chosen_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations might include removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# For analysis, identify a numeric field from the printed columns, e.g. 'MW' for molecular weight, use its @id

# We'll attempt to automatically select a numeric column from the dataframe
df = dataframes[chosen_rs_id]
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
if numeric_cols:
    numeric_field = numeric_cols[0]  # Use the first numeric column
    print(f"Using numeric column: {numeric_field}")
else:
    # If no numeric columns, pick the first column (may need to convert to numeric)
    numeric_field = df.columns[0]
    print(f"No numeric columns detected, using: {numeric_field}")
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Set filter threshold (10th percentile to allow filtering)
threshold = df[numeric_field].quantile(0.1)
filtered_df = df[df[numeric_field] > threshold]

print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric column
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a categorical field if available (not the numeric_field)
group_field = None
for col in df.columns:
    # Skip the numeric column, look for potentially categorical fields
    if col != numeric_field and df[col].nunique() < 10:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped {numeric_field} by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the chosen numeric field
plt.figure(figsize=(7,4))
df[numeric_field].hist(bins=30, grid=False)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouping was performed, plot bar chart
if group_field:
    plt.figure(figsize=(8,4))
    grouped_df_sorted = grouped_df.sort_values(numeric_field, ascending=False)
    plt.bar(grouped_df_sorted[group_field].astype(str), grouped_df_sorted[numeric_field])
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using the `mlcroissant` library. We reviewed available record sets, loaded data using `@id` fields, performed basic filtering and normalization on numerical columns, and produced simple visualizations. For deeper analysis, use the record set and field `@id`s to focus your queries and link documentation with your code. Explore the schema at the dataset URL for richer metadata integration and logic.